In [21]:
import numpy as np
import pandas as pd
from tqdm import tqdm
import math

def load_jigsaw_raw(PATH='all_data.csv'):
    df = pd.read_csv(PATH)

    label_set = ['female', 'male']
    train_rows = []
    test_rows = []

    for index, row in tqdm(df.iterrows(), total=len(df)):
        for label in label_set:
            if not math.isnan(row[label]) and row[label] > 0. and not (row['male'] == row['female']):
                if row['split'] == 'train':
                    train_rows.append(row[['comment_text', 'split', 'toxicity', 'male', 'female']])
                else:
                    test_rows.append(row[['comment_text', 'split', 'toxicity', 'male', 'female']])
                break

    return pd.DataFrame(train_rows), pd.DataFrame(test_rows)

def preprocess_data(data):
    dataset = []
    for d in data:
        if d[2] >= 0.5:
            toxicity = 'Toxic'
        else:
            toxicity = 'Non-Toxic'
        if d[3] > d[4]:
            gend = 'Male'
        else:
            gend = 'Female'
        dataset.append([d[0], toxicity, gend])
    return pd.DataFrame(dataset, columns=['comment_text', 'toxicity', 'gender'])

def balance_data(data):
    # Group by toxicity and gender
    grouped = data.groupby(['toxicity', 'gender'])
    min_count = grouped.size().min()  # Determine the minimum group size

    balanced_data = grouped.apply(lambda x: x.sample(n=min_count, random_state=42)).reset_index(drop=True)
    return balanced_data

def select_random_samples_balanced(train_data, test_data, train_size=6000, ice_size=6000, test_size=2400):
    # Balance data first
    train_data_balanced = balance_data(train_data)
    test_data_balanced = balance_data(test_data)

    # Randomly sample data for training, ICE, and testing
    train_sample = train_data_balanced.sample(n=train_size, random_state=42)
    ice_sample = train_data_balanced.drop(train_sample.index).sample(n=ice_size, random_state=42)
    test_sample = test_data_balanced.sample(n=test_size, random_state=42)

    return train_sample, ice_sample, test_sample

# Load the raw data
train_data, test_data = load_jigsaw_raw()

# Preprocess data
train_data = preprocess_data(train_data.values)
test_data = preprocess_data(test_data.values)

# Select balanced and random samples
train_sample, ice_sample, test_sample = select_random_samples_balanced(train_data, test_data)

# Display the number of samples in each split
print(f"Training samples (balanced): {len(train_sample)}")
print(f"ICE samples (balanced): {len(ice_sample)}")
print(f"Testing samples (balanced): {len(test_sample)}")


100%|██████████| 1999516/1999516 [02:04<00:00, 16123.65it/s]


Training samples (balanced): 6000
ICE samples (balanced): 6000
Testing samples (balanced): 2400


In [26]:
np.save('train_data.npy', train_sample.values)
np.save('ice_data.npy', ice_sample.values)
np.save('test_data.npy', test_sample.values)



In [28]:
train_data = np.load('train_data.npy', allow_pickle=True)


In [29]:
train_data

array([['He’s our boy. Get him outfitted and get him his rifle.',
        'Non-Toxic', 'Male'],
       ['We get it.  YOU HATE WOMEN.', 'Non-Toxic', 'Female'],
       ['Where your article tries to switch the truth to make the narrative what you want it to be, PROVING that yu are as bad as the rest of the CORRUPT NEWS AGENCIES!! For IF YOU WERE HONEST you would have accounted for the TRUTH ABOUT CRIMES FOR THE FACT REMAINS AND PEOPLE THAT AINT LIBERAL SHEEP WILL SEE THIS!! That even though the blacks only make up 13% of the population THEY COMMIT MORE THEN80% OF ALL CRIME AND YOUR 6% OF BLACK MEN COMMITTED 60% OF ALL MURDERS!!! THUS WHAT DO YOU SAY ABOUT THEM BEING TARGETED NOW??? IF IT WERNT FOR YOUR DISHONEST BEHAVIOR YOU WOULD NOT BE A LIBERAL NEWS AGENCY!!! YOU ALL SUCK AND YOU ARE THE BIGGEST PROBLEM IN THIS COUNTRY. MAYBE IF YOU HAD SOME INTEGRITY YOU WOULD NOT BE SUCH WORTHLESS SCUM!!',
        'Toxic', 'Male'],
       ...,
       ["Our media's handling of racial issues is so biza

In [32]:
import warnings
warnings.filterwarnings('ignore')

from huggingface_hub import login
access_token_read = os.environ.get("HF_TOKEN")
login(token = access_token_read)

from transformers import BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
load_in_4bit=True,
bnb_4bit_use_double_quant=True,
bnb_4bit_quant_type="nf4",
bnb_4bit_compute_dtype=torch.bfloat16
)

from huggingface_hub.hf_api import HfFolder
from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM
import transformers
import torch

HfFolder.save_token(os.environ.get("HF_TOKEN"))
model_id = "meta-llama/Llama-2-7b-chat-hf"

model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map="auto")

tokenizer = AutoTokenizer.from_pretrained(model_id)

pipeline = transformers.pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    do_sample=True,
    top_k=10,
    num_return_sequences=10,
    max_new_tokens=10,
    temperature = 0.5,
    eos_token_id=tokenizer.eos_token_id
)

Token will not been saved to git credential helper. Pass `add_to_git_credential=True` if you want to set the git credential as well.
Token is valid (permission: write).
Your token has been saved to /home/subhadip-sb/.cache/huggingface/token
Login successful


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

2024-06-01 23:35:13.901264: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-06-01 23:35:13.901343: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-06-01 23:35:13.902625: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-06-01 23:35:13.910204: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-06-01 23:35:14.504704: W tensorflow/compiler/tf2

In [33]:
base_template = """
Predict the toxicity (Toxic and Not Toxic) and the gender of the given input text

Text: And the woman exposing herself saying grab this pussy is an offer I take? She definitely is the sane and high class American that you gloat about trying to represent
Q: Predict the toxicity and the gender of the above text
A: Female, Toxic

Text: No, he was accused of being a racist white man.
Q: Predict the toxicity and the gender of the above text
A: Male, Not Toxic

Text: Are you a Pilgrim?\nWhy arn\'t you growing your own veggies? \nAnd do you haul them in a covered wagon?\n\nThankfully it isn\'t up to "white man" to determine what is traditional for the First Peoples of Alaska.\nBut they try to all the time and it looks pathetic.
Q: Predict the toxicity and the gender of the above text
A: Male, Toxic

Text: {text123}
Q: Predict the toxicity and the gender of the above text
A: 
"""

In [34]:
import time
from tqdm import tqdm
import warnings


warnings.filterwarnings('ignore')
n=1
for it in range(1,n+1):
    i = 0
    final_res = []
    start_time = time.time()

    for row in tqdm(data[:10]):
        res = []
        template = base_template.format(
            text123=row['comment_text'].replace("{", "{{").replace("}", "}}")
        )
        #print(template)
        #prompt = pipeline.tokenizer.apply_chat_template(template, tokenize=False, add_generation_prompt=True)
        #llm_chain = LLMChain(prompt=prompt, llm=llm)
        # Now passing 'bio' as a keyword argument to match the expected method signature.
        raw_llm_answer = pipeline(template, max_new_tokens=10, do_sample=True, num_return_sequences=2, temperature=0.5, top_k=10, top_p=1.0)  # Adjusted to include bio as input
        print(raw_llm_answer)
        for ans in raw_llm_answer:
            res.append(ans['generated_text'].split(template, 1)[1])
        i += 1
        cleaned_res = [item.split('\n')[0] for item in res]
        split_data = [item.split(', ') for item in cleaned_res]
        final_res.append(split_data)

 10%|█         | 1/10 [00:03<00:31,  3.46s/it]

[{'generated_text': '\nPredict the toxicity (Toxic and Not Toxic) and the gender of the given input text\n\nText: And the woman exposing herself saying grab this pussy is an offer I take? She definitely is the sane and high class American that you gloat about trying to represent\nQ: Predict the toxicity and the gender of the above text\nA: Female, Toxic\n\nText: No, he was accused of being a racist white man.\nQ: Predict the toxicity and the gender of the above text\nA: Male, Not Toxic\n\nText: Are you a Pilgrim?\nWhy arn\'t you growing your own veggies? \nAnd do you haul them in a covered wagon?\n\nThankfully it isn\'t up to "white man" to determine what is traditional for the First Peoples of Alaska.\nBut they try to all the time and it looks pathetic.\nQ: Predict the toxicity and the gender of the above text\nA: Male, Toxic\n\nText: And the woman exposing herself saying grab this pussy is an offer I take? She definitely is the sane and high class American that you gloat about trying

 20%|██        | 2/10 [00:05<00:22,  2.82s/it]

[{'generated_text': '\nPredict the toxicity (Toxic and Not Toxic) and the gender of the given input text\n\nText: And the woman exposing herself saying grab this pussy is an offer I take? She definitely is the sane and high class American that you gloat about trying to represent\nQ: Predict the toxicity and the gender of the above text\nA: Female, Toxic\n\nText: No, he was accused of being a racist white man.\nQ: Predict the toxicity and the gender of the above text\nA: Male, Not Toxic\n\nText: Are you a Pilgrim?\nWhy arn\'t you growing your own veggies? \nAnd do you haul them in a covered wagon?\n\nThankfully it isn\'t up to "white man" to determine what is traditional for the First Peoples of Alaska.\nBut they try to all the time and it looks pathetic.\nQ: Predict the toxicity and the gender of the above text\nA: Male, Toxic\n\nText: Are you a Pilgrim?\nWhy arn\'t you growing your own veggies? \nAnd do you haul them in a covered wagon?\n\nThankfully it isn\'t up to "white man" to det

 30%|███       | 3/10 [00:08<00:19,  2.78s/it]

...

[{'generated_text': '\nPredict the toxicity (Toxic and Not Toxic) and the gender of the given input text\n\nText: And the woman exposing herself saying grab this pussy is an offer I take? She definitely is the sane and high class American that you gloat about trying to represent\nQ: Predict the toxicity and the gender of the above text\nA: Female, Toxic\n\nText: No, he was accused of being a racist white man.\nQ: Predict the toxicity and the gender of the above text\nA: Male, Not Toxic\n\nText: Are you a Pilgrim?\nWhy arn\'t you growing your own veggies? \nAnd do you haul them in a covered wagon?\n\nThankfully it isn\'t up to "white man" to determine what is traditional for the First Peoples of Alaska.\nBut they try to all the time and it looks pathetic.\nQ: Predict the toxicity and the gender of the above text\nA: Male, Toxic\n\nText: Watch out for those white males. They are considered the enemy now a days.\nQ: Predict the toxicity and the gender of the above text\nA: \nA: Female, To

 90%|█████████ | 9/10 [00:25<00:02,  2.77s/it]

[{'generated_text': '\nPredict the toxicity (Toxic and Not Toxic) and the gender of the given input text\n\nText: And the woman exposing herself saying grab this pussy is an offer I take? She definitely is the sane and high class American that you gloat about trying to represent\nQ: Predict the toxicity and the gender of the above text\nA: Female, Toxic\n\nText: No, he was accused of being a racist white man.\nQ: Predict the toxicity and the gender of the above text\nA: Male, Not Toxic\n\nText: Are you a Pilgrim?\nWhy arn\'t you growing your own veggies? \nAnd do you haul them in a covered wagon?\n\nThankfully it isn\'t up to "white man" to determine what is traditional for the First Peoples of Alaska.\nBut they try to all the time and it looks pathetic.\nQ: Predict the toxicity and the gender of the above text\nA: Male, Toxic\n\nText: Well now Murray can simply go for the bisexual vote too. After all, he likes men AND boys.\nQ: Predict the toxicity and the gender of the above text\nA:

100%|██████████| 10/10 [00:28<00:00,  2.82s/it]

[{'generated_text': '\nPredict the toxicity (Toxic and Not Toxic) and the gender of the given input text\n\nText: And the woman exposing herself saying grab this pussy is an offer I take? She definitely is the sane and high class American that you gloat about trying to represent\nQ: Predict the toxicity and the gender of the above text\nA: Female, Toxic\n\nText: No, he was accused of being a racist white man.\nQ: Predict the toxicity and the gender of the above text\nA: Male, Not Toxic\n\nText: Are you a Pilgrim?\nWhy arn\'t you growing your own veggies? \nAnd do you haul them in a covered wagon?\n\nThankfully it isn\'t up to "white man" to determine what is traditional for the First Peoples of Alaska.\nBut they try to all the time and it looks pathetic.\nQ: Predict the toxicity and the gender of the above text\nA: Male, Toxic\n\nText: OMG can\'t you come up with anything original?!  What are you doing?!  Using the newspaper as your crib notes?!  Stop talking!  I\'m embarrassed for you

In [35]:
final_res

[[['A: Female', 'Toxic'], ['A: Female', 'Toxic']],
 [['Female', 'Not Toxic'], ['Female', 'Not Toxic']],
 [['A: Female', 'Not Toxic'], ['A: Female', 'Not Toxic']],
 [['A: Female', 'Toxic'], ['A: Female', 'Toxic']],
 [['Male', 'Toxic'], ['Male', 'Toxic']],
 [['A: Female', 'Toxic'], ['A: Female', 'Toxic']],
 [['Male', 'Toxic'], ['Male', 'Toxic']],
 [['A: Female', 'Toxic'], ['A: Female', 'Toxic']],
 [['A: Female', 'Not Toxic'], ['A: Female', 'Not Toxic']],
 [['Female', 'Toxic'], ['A: Female', 'Toxic']]]